In [ ]:
!pip install -q unsloth
!pip install -q --no-deps "trl>=0.11" "datasets>=2.19"

In [ ]:
CONFIG = {
    "model_name": "unsloth/Qwen3-4B",
    "max_seq_length": 2048,
    "load_in_4bit": True,
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "epochs": 1,
    "train_samples": 2000,
    "batch_size": 2,
    "grad_accum": 4,
    "learning_rate": 2e-4,
    "warmup_steps": 10,
    "eval_samples": 200,
    "eval_max_new_tokens": 768,
    "output_dir": "outputs/qwen3-4b-gsm8k-lora",
    "seed": 3407,
}

In [ ]:
import transformers
transformers.logging.set_verbosity_error()

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
)
print("Model loaded:", CONFIG["model_name"])

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")
train_raw = gsm8k["train"]
test_raw = gsm8k["test"]
print(f"Train: {len(train_raw)} | Test: {len(test_raw)}")

In [ ]:
import re

def extract_gold(answer_text):
    return answer_text.split("####")[-1].strip().replace(",", "")

_BOXED_RE = re.compile(r"\\boxed\{([^{}]*)\}")
_NUM_RE = re.compile(r"[-+]?\d[\d,]*\.?\d*")

def _last_number(s):
    matches = _NUM_RE.findall(s)
    if not matches:
        return None
    return matches[-1].replace(",", "").rstrip(".")

def extract_pred(model_text):
    boxed = _BOXED_RE.findall(model_text)
    if boxed:
        n = _last_number(boxed[-1])
        if n is not None:
            return n
    if "####" in model_text:
        n = _last_number(model_text.split("####")[-1])
        if n is not None:
            return n
    return _last_number(model_text)

def numbers_match(pred, gold):
    if pred is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 1e-4
    except ValueError:
        return str(pred) == str(gold)

In [ ]:
SYSTEM = (
    "You are a helpful math tutor. Solve the problem step by step, concisely. "
    "End your response with the final numeric answer in the form \\boxed{answer}."
)

def build_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

In [ ]:
def evaluate(model, n_samples, max_new_tokens, verbose_wrong=0):
    FastLanguageModel.for_inference(model)
    correct = 0
    wrong_shown = 0
    subset = test_raw.select(range(n_samples))

    for i, row in enumerate(subset):
        prompt = build_prompt(row["question"])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        pred = extract_pred(gen)
        gold = extract_gold(row["answer"])
        ok = numbers_match(pred, gold)
        correct += ok

        if not ok and wrong_shown < verbose_wrong:
            wrong_shown += 1
            print(f"\n--- WRONG #{wrong_shown} ---")
            print("Q:", row["question"][:200])
            print("Model said:", pred, "| Gold:", gold)
            print("Output tail:", gen[-300:])

        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{n_samples}  running acc={correct/(i+1):.3f}")

    acc = correct / n_samples
    print(f"\nAccuracy: {acc:.3f} ({correct}/{n_samples})")
    return acc

In [ ]:
print("=== BASELINE (stock model, no fine-tuning) ===")
baseline_acc = evaluate(
    model,
    n_samples=CONFIG["eval_samples"],
    max_new_tokens=CONFIG["eval_max_new_tokens"],
    verbose_wrong=5,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

In [ ]:
def to_training_text(row):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["answer"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, enable_thinking=False
    )
    return {"text": text}

train_ds = (
    train_raw.select(range(CONFIG["train_samples"]))
    .map(to_training_text, remove_columns=train_raw.column_names)
)
print(train_ds[0]["text"][:800])

In [ ]:
from trl import SFTTrainer, SFTConfig

try:
    FastLanguageModel.for_training(model)
except AttributeError:
    model.train()
    model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=CONFIG["max_seq_length"],
        per_device_train_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["grad_accum"],
        num_train_epochs=CONFIG["epochs"],
        learning_rate=CONFIG["learning_rate"],
        warmup_steps=CONFIG["warmup_steps"],
        logging_steps=10,
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        seed=CONFIG["seed"],
        output_dir=CONFIG["output_dir"],
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("Training done.")

In [ ]:
print("=== AFTER FINE-TUNING ===")
finetuned_acc = evaluate(
    model,
    n_samples=CONFIG["eval_samples"],
    max_new_tokens=CONFIG["eval_max_new_tokens"],
    verbose_wrong=5,
)

print("\n================ RESULT ================")
print(f"Baseline:    {baseline_acc:.3f}")
print(f"Fine-tuned:  {finetuned_acc:.3f}")
print(f"Delta:       {finetuned_acc - baseline_acc:+.3f}")
print("=======================================")

In [ ]:
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print("Saved adapters to:", CONFIG["output_dir"])

In [ ]:
def solve(question, max_new_tokens=768, show_full=True):
    FastLanguageModel.for_inference(model)
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    if show_full:
        print(gen)
        print("\n--- extracted answer:", extract_pred(gen), "---")
    return extract_pred(gen)

In [ ]:
solve("A shop sells pens at $3 each. If I buy 4 pens and pay with a $20 note, how much change do I get?")